### 프로덕션 레벨 ReAct 워크플로우 구축 (LangGraph)
- LangGraph를 활용하여 상태(State) 기반의 에이전트 워크플로우를 설계한다.
- 순환(Cyclic) 그래프 구조를 통해 모델이 원하는 결과를 낼 때까지 스스로 도구를 호출하고 검증하는 로직을 구현한다.

### 1. 상태(State) 및 도구 정의
LangGraph는 각 노드(Node) 간에 전달되는 상태(State)를 정의해야 합니다. 에이전트의 상태는 주고받은 메시지들의 목록입니다.

In [1]:
import os
import operator
from typing import TypedDict, Annotated, Sequence
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage,ToolMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

In [2]:
# 1. 상태객체(state object) 정의
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

# 2. 도구 정의
@tool
def get_current_time(loaction:str)->str:
    '''특정 지역의 현재 시간을 반환합니다.'''
    import datetime
    return f"{loaction}의 현재 시간은 {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} 입니다"

tools = [get_current_time]
# langgraph.prebuilt 의 ToolNode 를 사용하면 실행노드를 쉽게 만들수 있음
tool_node = ToolNode(tools)
print('상태 정의 및 도구 초기화 완료')

상태 정의 및 도구 초기화 완료


### 2. 에이전트 노드 및 라우팅 함수 구현
모델을 호출하는 노드(Node)와, 모델의 응답에 따라 도구를 실행할지 종료할지 결정하는 엣지(Edge) 로직을 만듭니다.

In [3]:
# 1. 모델과 도구 바인딩(Function calling)
model = ChatOpenAI(api_key=os.getenv('OPENAI_API_KEY'),model='gpt-5.4-nano',temperature=0)
model_with_tools = model.bind_tools(tools)

# 2. 에이전트 노드 함수 : 상태를 받아 모델을 호출하고 상태를 업데이트
def call_model(state:AgentState):
    messages = state['messages']
    response = model_with_tools.invoke(messages)
    return {'messages':[response]}

# 3.라우팅 로직 : 모델이 도구 호출(tool_calls)을 요청했는지 확인
def should_continue(state:AgentState):
    messages = state['messages']
    last_message = messages[-1]
    # 마지막 메세지에 도구 호출 정보가 있다면 도구 노드로 이동
    if last_message.tool_calls:
        return 'continue'
    # 도구 호출이 없다면(최종답변을 도출했다면) 종료
    return 'end'

print('에이전트 노드와 라우터 준비')

에이전트 노드와 라우터 준비


### 3. StateGraph 워크플로우 조립 및 컴파일
노드와 엣지를 연결하여 전체 ReAct 사이클(추론 -> 도구실행 -> 관찰 -> 추론...)을 완성합니다.

In [4]:
# 워크플로우 그래프 생성  # 노드(node) 엣지(edge) 흐름(flow)
workflow = StateGraph(AgentState)
# 노드 등록
workflow.add_node('agent', call_model)  # 추론 노드
workflow.add_node('action',tool_node)  # 행동(도구) 노드

# 시작점을 agent로 설정
workflow.set_entry_point('agent')

# workflow.add_edge('a','b')  # 무조건 a->b
# 조건부 엣지 등록 : agent의 결과에 따라서 분기
workflow.add_conditional_edges(
    'agent',   # 현재노드 이름
    should_continue,  #라우터 : 분기판단
    {
        'continue':'action',   # 라우터가 continue를 반환하면 action 노드로 이동
        'end':END   # 종료
    }
)

# 일반 엣지 등록 : action(도구 실행)이 끝나면 무조건 agent(추론)으로 다시 돌아가서 관찰결과 전달
workflow.add_edge('action','agent')

# 그래프 컴파일
app = workflow.compile()

print('ReAct 워크플로우 컴파일 완료')

ReAct 워크플로우 컴파일 완료


### 4. 워크플로우 실행 및 스트리밍
구성된 그래프에 초기 상태(사용자 질문)를 입력하고 실행 흐름을 단계별로 추적해봅니다.

In [ ]:
inputs = {'messages' : [HumanMessage(content='서울의 현재  시간은 언제인가요?')]}
print(' == LangGraph ReAct 실행 흐름 ==')

for output in app.stream(inputs):
    for node_name, state_update in output.items():
        print(f"\n[노드 실행 완료]: {node_name}")
        
        # 업데이트된 마지막 메시지 확인
        latest_message = state_update['messages'][-1]
        
        if isinstance(latest_message, AIMessage):
            if hasattr(latest_message, "tool_calls") and latest_message.tool_calls:
                tool_call = latest_message.tool_calls[0]
                print(f" -> Thought: 도구 '{tool_call['name']}' 호출 결정")
            else:
                print(f" -> Final Answer: {latest_message.content}")
        elif isinstance(latest_message, ToolMessage):
            print(f" -> Observation: {latest_message.content}")

print("\n=== 실행 종료 ===")


 == LangGraph ReAct 실행 흐름 ==
{'agent': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 138, 'total_tokens': 157, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-Dkjw51mNjDygvBJNSqvVX5fLEFqR2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e722d-5b94-7663-8349-a4416c71c43d-0', tool_calls=[{'name': 'get_current_time', 'args': {'loaction': '서울'}, 'id': 'call_7uV9imC9MBJjfV31wH6bwk7a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 138, 'output_tokens': 19, 'total_tokens': 157, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_det